# Get datasets


In [82]:
import pandas as pd
import numpy as np

wines = pd.read_csv('winemag-data-130k-v2.csv', index_col=0)

wines.head()

,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm
3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,Lake Michigan Shore,NaN,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian
4,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks


In [83]:
import re

def extract_vintage(title):
    matches = re.findall(r'\b(19[0-9]{2}|20[0-2][0-9]|2025)\b', title)
    if not matches:
        return np.nan
    else:
        years = map(int, matches)
        return max(years)


wines['year'] = wines['title'].apply(extract_vintage)

# Normalize title by removing vintage
def normalize_title(title, year):
    if np.isnan(year):
        return title
    else:
        return title.replace(str(int(year)), '')

wines['normalized_title'] = wines.apply(lambda row: normalize_title(row['title'], row['year']), axis=1)

# Find the latest vintage per normalized title
latest_vintage_map = wines.groupby('normalized_title')['year'].max()

# Filter rows where vintage matches the latest for that normalized title
df_latest = wines[wines.apply(lambda row: row['year'] == latest_vintage_map[row['normalized_title']], axis=1)]

# Drop helper column
wines = df_latest.drop(columns=['normalized_title'])

# View result
print(wines['title'].nunique())
wines.head()


78123


,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery,year
0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia,2013.0
2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm,2013.0
3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,Lake Michigan Shore,NaN,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian,2013.0
4,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks,2012.0
5,Spain,Blackberry and raspberry aromas show a typical...,Ars In Vitro,87,15.0,Northern Spain,Navarra,NaN,Michael Schachner,@wineschach,Tandem 2011 Ars In Vitro Tempranillo-Merlot (N...,Tempranillo-Merlot,Tandem,2011.0


In [85]:
wines_bordeaux = pd.read_csv('BordeauxWines.csv') #https://www.kaggle.com/datasets/mexwell/21st-century-bordeaux-wine-dataset

core_columns = ['Name', 'Score', 'Year', 'Price']

flavor_cols = [col for col in wines_bordeaux.columns if col not in core_columns]

wines_bordeaux["description"] = wines_bordeaux[flavor_cols].apply(
    lambda row: [col for col, val in row.items() if val == 1], axis=1
)
df_bordeaux = wines_bordeaux[core_columns + ["description"]].copy()

df_bordeaux.rename(columns={'Name': 'title', 'Score': 'points', 'Year': 'year', 'Price': 'price'}, inplace=True)

df_bordeaux['price'] = df_bordeaux['price'].str.replace('$', '', regex=False)

df_bordeaux['price'] = pd.to_numeric(df_bordeaux['price'], errors='coerce')

df_bordeaux['description'] = df_bordeaux['description'].apply(tuple)

df_bordeaux.head()

,title,points,year,price,description
0,ChÃ¢teau Croix Figeac St.-Emilion,84,2008,20.0,"(BLACK CHERRY, TANNINS_LOW, LIGHT-BODIED, RED,..."
1,ChÃ¢teau Fonroque St.-Emilion,84,2008,29.0,"(KIRSCH, ANISE, TANNINS_HIGH, RED, WELL-STRUCT..."
2,ChÃ¢teau Grand Bertin de St.-Clair MÃ©doc,84,2008,NaN,"(BLACK CHERRY, LEAF, TANNINS_HIGH, RED, FLAVOR..."
3,ChÃ¢teau Lion Beaulieu Bordeaux White,84,2008,NaN,"(PEAR, LIGHT-BODIED, SOLID, WHITE, COMPACT, CO..."
4,ChÃ¢teau Marsau CÃ´tes de Francs,84,2008,20.0,"(CHERRY, LEAF, FRAME, LIGHT-BODIED, RED, ACIDI..."


In [86]:
from ftfy import fix_text

df_bordeaux['title'] = df_bordeaux['title'].apply(lambda x: fix_text(x) if isinstance(x, str) else x)
df_bordeaux.head()

,title,points,year,price,description
0,Château Croix Figeac St.-Emilion,84,2008,20.0,"(BLACK CHERRY, TANNINS_LOW, LIGHT-BODIED, RED,..."
1,Château Fonroque St.-Emilion,84,2008,29.0,"(KIRSCH, ANISE, TANNINS_HIGH, RED, WELL-STRUCT..."
2,Château Grand Bertin de St.-Clair Médoc,84,2008,NaN,"(BLACK CHERRY, LEAF, TANNINS_HIGH, RED, FLAVOR..."
3,Château Lion Beaulieu Bordeaux White,84,2008,NaN,"(PEAR, LIGHT-BODIED, SOLID, WHITE, COMPACT, CO..."
4,Château Marsau Côtes de Francs,84,2008,20.0,"(CHERRY, LEAF, FRAME, LIGHT-BODIED, RED, ACIDI..."


In [87]:
# Find the latest vintage per normalized title
latest_vintage_map = df_bordeaux.groupby('title')['year'].max()

# Filter rows where vintage matches the latest for that normalized title
df_bordeaux = df_bordeaux[df_bordeaux.apply(lambda row: row['year'] == latest_vintage_map[row['title']], axis=1)]

# View result
print(df_bordeaux['title'].nunique())
df_bordeaux.head()

3433


,title,points,year,price,description
19,Château Franc-Maillet Pomerol,84,2009,25.0,"(PLUM, PRUNE, RAISIN, TANNINS_LOW, LIGHT-BODIE..."
21,Château Penin Sauvignon Bordeaux,85,2007,NaN,"(APPLE, WHITE, EXCELLENT FINISH, FLAVORS, FOCU..."
61,Château Clos Chaumont Côtes de Bordeaux,87,2008,NaN,"(CHERRY, PLUM, LEAF, TANNINS_MEDIUM, MEDIUM-BO..."
117,Clos Bertineau Montagne-St.-Emilion,89,2009,NaN,"(BLACKBERRY, PLUM, TANNINS_HIGH, LIGHT-BODIED,..."
122,Château Dassault St.-Emilion Le D de Dassault,89,2009,NaN,"(RASPBERRY, CHERRY, TANNINS_HIGH, RED, SOLID, ..."


In [88]:
wines_2 = pd.read_csv('wine-ratings.csv') #https://github.com/paiml/wine-ratings/blob/main/wine-ratings.csv

wines_2.drop('grape', axis=1, inplace=True)
wines_2.drop('Unnamed: 0', axis=1, inplace=True)

wines_2.rename(columns={'name': 'title', 'rating': 'points', 'region': 'province', 'notes': 'description'}, inplace=True)

wines_2.head()

,title,province,variety,points,description
0,1000 Stories Bourbon Barrel Aged Batch Blue Ca...,"Mendocino, California",Red Wine,91.0,"This is a very special, limited release of 100..."
1,1000 Stories Bourbon Barrel Aged Gold Rush Red...,California,Red Wine,89.0,The California Gold Rush was a period of coura...
2,1000 Stories Bourbon Barrel Aged Gold Rush Red...,California,Red Wine,90.0,The California Gold Rush was a period of coura...
3,1000 Stories Bourbon Barrel Aged Zinfandel 2013,"North Coast, California",Red Wine,91.0,"The wine has a deep, rich purple color. An int..."
4,1000 Stories Bourbon Barrel Aged Zinfandel 2014,California,Red Wine,90.0,Batch #004 is the first release of the 2014 vi...


In [90]:
wines_2['year'] = wines_2['title'].apply(extract_vintage)

# Normalize title by removing vintage
def normalize_title(title, year):
    if np.isnan(year):
        return title
    else:
        return title.replace(str(int(year)), '')

wines_2['normalized_title'] = wines_2.apply(lambda row: normalize_title(row['title'], row['year']), axis=1)

# Find the latest vintage per normalized title
latest_vintage_map = wines_2.groupby('normalized_title')['year'].max()

# Filter rows where vintage matches the latest for that normalized title
wines_2_latest = wines_2[wines_2.apply(lambda row: row['year'] == latest_vintage_map[row['normalized_title']], axis=1)]

# Drop helper column
wines_2 = wines_2_latest.drop(columns=['normalized_title'])

# View result
print(wines_2['title'].nunique())
wines_2.head()

12496


,title,province,variety,points,description,year
0,1000 Stories Bourbon Barrel Aged Batch Blue Ca...,"Mendocino, California",Red Wine,91.0,"This is a very special, limited release of 100...",2016
2,1000 Stories Bourbon Barrel Aged Gold Rush Red...,California,Red Wine,90.0,The California Gold Rush was a period of coura...,2017
6,1000 Stories Bourbon Barrel Aged Zinfandel 2017,California,Red Wine,92.0,"Batch 55 embodies an opulent vintage, which sa...",2017
7,12 Linajes Crianza 2014,"Ribera del Duero, Spain",Red Wine,92.0,Red with violet hues. The aromas are very inte...,2014
8,12 Linajes Reserva 2012,"Ribera del Duero, Spain",Red Wine,94.0,"On the nose, a complex predominance of mineral...",2012


## Combine datasets

In [91]:
combined_wines = pd.concat([wines, df_bordeaux, wines_2], ignore_index=True)
combined_wines.head()

,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery,year
0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87.0,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia,2013.0
1,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87.0,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm,2013.0
2,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87.0,13.0,Michigan,Lake Michigan Shore,NaN,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian,2013.0
3,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87.0,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks,2012.0
4,Spain,Blackberry and raspberry aromas show a typical...,Ars In Vitro,87.0,15.0,Northern Spain,Navarra,NaN,Michael Schachner,@wineschach,Tandem 2011 Ars In Vitro Tempranillo-Merlot (N...,Tempranillo-Merlot,Tandem,2011.0


## Remove duplicate rows

In [92]:
print(f'Dataframe size: {combined_wines.shape}')
print(f'{combined_wines.duplicated().sum()} duplicate rows')

wines = combined_wines.drop_duplicates()
print(f'New dataframe size: {wines.shape}')

Dataframe size: (101203, 14)
6885 duplicate rows
New dataframe size: (94318, 14)


## Drop rows without description

In [93]:
wines = wines[wines['description'].notna() & wines['description'].str.strip().astype(bool)]
print(f'New dataframe size: {wines.shape}')

New dataframe size: (94263, 14)


## Drop expensive and old wines

In [94]:
wines['price'].describe()

count    75527.000000
mean        34.563600
std         39.239137
min          1.000000
25%         17.000000
50%         25.000000
75%         40.000000
max       3300.000000
Name: price, dtype: float64

In [95]:
wines = wines[wines['price'] <= 100]

In [96]:
wines['year'].describe()

count    73454.000000
mean      2011.107128
std          3.777543
min       1904.000000
25%       2009.000000
50%       2012.000000
75%       2014.000000
max       2017.000000
Name: year, dtype: float64

In [97]:
wines = wines[wines['year'] > 2005]

## Drop wines with low rating points

In [98]:
wines['points'].describe()

count    68023.000000
mean        88.166003
std          2.976023
min         60.000000
25%         86.000000
50%         88.000000
75%         90.000000
max         99.000000
Name: points, dtype: float64

In [99]:
wines = wines[wines['points'] >= 85] # Wine with at least 85 points considered "good" (https://en.wikipedia.org/wiki/Wine_rating , https://www.wineenthusiast.com/ratings/ )


## Drop columns and clean description data

In [100]:
import nltk
import string
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))


stemmer = nltk.PorterStemmer()
translator = str.maketrans('', '', string.punctuation)

cleaned = wines.drop(columns=['region_1', 'region_2', 'taster_twitter_handle', 'taster_name', 'winery'])

cleaned['description'] = cleaned['description'].apply(
    lambda x: ' '.join(map(str, x)) if isinstance(x, (tuple, list)) else str(x)
)

def clean_descriptions(description: str):
    if not isinstance(description, str):
        if isinstance(description, (tuple, list)):
            description = ' '.join(map(str, description))
        else:
            description = str(description)
    
    puncs_removed = description.translate(translator)
    tokens = puncs_removed.lower().split()
    stemmed_tokens = list(map(stemmer.stem, tokens))
    stop_words_removed = ' '.join([token for token in stemmed_tokens if token not in stop_words])
    return stop_words_removed

cleaned['taste_profile'] = cleaned['description'].apply(clean_descriptions)
cleaned.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/kallevapaa/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,country,description,designation,points,price,province,title,variety,year,taste_profile
1,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87.0,14.0,Oregon,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,2013.0,tart snappi flavor lime flesh rind domin green...
2,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87.0,13.0,Michigan,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,2013.0,pineappl rind lemon pith orang blossom start a...
3,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87.0,65.0,Oregon,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,2012.0,much like regular bottl 2012 thi come across r...
4,Spain,Blackberry and raspberry aromas show a typical...,Ars In Vitro,87.0,15.0,Northern Spain,Tandem 2011 Ars In Vitro Tempranillo-Merlot (N...,Tempranillo-Merlot,2011.0,blackberri raspberri aroma show typic navarran...
6,US,Building on 150 years and six generations of w...,NaN,87.0,12.0,California,Mirassou 2012 Chardonnay (Central Coast),Chardonnay,2012.0,build 150 year six gener winemak tradit wineri...


## Combine rows containing data for same wine

In [101]:
for col in ['points', 'price']:
    count = cleaned[col].apply(lambda x: isinstance(x, (str))).sum()
    print(f"{col}: {count} rows are strings")

points: 0 rows are strings
price: 0 rows are strings


In [102]:
agg_funcs = {
    'country': 'first',
    'description': lambda x: ' '.join(x),
    'designation': 'first',
    'points': 'mean',
    'price': 'mean',
    'province': 'first',
    'variety': 'first',
    'year': 'first',
    'taste_profile': lambda x: ' '.join(x)
}

aggregated = cleaned.groupby('title', as_index=False).agg(agg_funcs)
aggregated.head()

,title,country,description,designation,points,price,province,variety,year,taste_profile
0,10 Knots 2006 Chardonnay (Santa Barbara County),US,Oaky influences give this wine a candied taste...,None,85.0,21.0,California,Chardonnay,2006.0,oaki influenc give thi wine candi tast caramel...
1,100 Percent Wine 2012 All Profits to Charity R...,US,"Juicy and fresh, this deeply colored wine offe...",All Profits to Charity,89.0,18.0,California,Red Blend,2012.0,juici fresh thi deepli color wine offer lot gr...
2,100 Percent Wine 2015 Moscato (California),US,"Sweet and light bodied, this wine has plenty o...",None,86.0,18.0,California,Moscato,2015.0,sweet light bodi thi wine ha plenti jasmin aro...
3,1000 Stories 2013 Bourbon Barrel Aged Zinfande...,US,This approachable wine from the Fetzer organiz...,Bourbon Barrel Aged,91.0,19.0,California,Zinfandel,2013.0,thi approach wine fetzer organ wa age bourbon ...
4,1000 Stories 2014 Bourbon Barrel Aged Batch No...,US,Exotically fruity with an enticing floral char...,Bourbon Barrel Aged Batch No 13,90.0,19.0,California,Zinfandel,2014.0,exot fruiti entic floral charact thi fullbodi ...


View aggregation result for the most frequently occurring wine title:

In [103]:
title_mode = cleaned['title'].mode()[0]
cleaned[cleaned['title'] == title_mode]

,country,description,designation,points,price,province,title,variety,year,taste_profile
16866,Portugal,This big and bold wine exudes power and fruiti...,Bridão Reserva,91.0,14.0,Tejo,Adega Cooperativa do Cartaxo 2011 Bridão Reser...,Portuguese Red,2011.0,thi big bold wine exud power fruiti ha ripe fr...
22680,Portugal,"Firmly structured, this has both great tannins...",Bridão Reserva,91.0,15.0,Tejo,Adega Cooperativa do Cartaxo 2011 Bridão Reser...,Portuguese Red,2011.0,firmli structur thi ha great tannin equal big ...


In [104]:
aggregated[aggregated['title'] == title_mode]

,title,country,description,designation,points,price,province,variety,year,taste_profile
493,Adega Cooperativa do Cartaxo 2011 Bridão Reser...,Portugal,This big and bold wine exudes power and fruiti...,Bridão Reserva,91.0,14.5,Tejo,Portuguese Red,2011.0,thi big bold wine exud power fruiti ha ripe fr...


## Wine recommender

In [105]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

vectorizer = TfidfVectorizer(
    max_df=0.5,
    min_df=5,
    ngram_range=(1,2)
)

tfidf_matrix = vectorizer.fit_transform(aggregated['taste_profile'])
tfidf_matrix.shape


(60445, 53724)

# Dump vectorizer and TF-IDF matrix to files so they can be used by our API-endpoint

In [106]:
import joblib
from scipy import sparse

joblib.dump(vectorizer, "../assets/tfidf_vectorizer.joblib", compress=3)
sparse.save_npz("../assets/tfidf_matrix.npz", tfidf_matrix)
aggregated.to_csv('../assets/wines_data.csv')


### User input is words describing taste

In [107]:
def recommend_wine_by_description(description, vectorizer=vectorizer, tfidf_matrix=tfidf_matrix):
    translator = str.maketrans('', '', string.punctuation)
    description = description.translate(translator)
    tokens = description.lower().split()
    stemmed = list(map(stemmer.stem, tokens))
    description = ' '.join(stemmed)

    tfidf_input = vectorizer.transform([description])
    cosine_sim = linear_kernel(tfidf_matrix, tfidf_input)

    sim_scores = list(enumerate(cosine_sim))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[0:5]
    position = [i[0] for i in sim_scores]

    return aggregated.iloc[position]

user_input = 'white flowers, mineral, peach'
recommendations = recommend_wine_by_description(user_input)
recommendations

,title,country,description,designation,points,price,province,variety,year,taste_profile
9402,Cascata 2006 Cascade Riesling (Finger Lakes),US,"This wine offers aromas of white flowers, spic...",Cascade,85.0,16.0,New York,Riesling,2006.0,thi wine offer aroma white flower spice citru ...
21259,Dowsett Family 2014 Aunt Diane's Vineyard Ries...,US,"Perfumed aromas offer notes of white flowers, ...",Aunt Diane's Vineyard,88.0,16.0,Washington,Riesling,2014.0,perfum aroma offer note white flower miner pin...
8764,Carmel Road 2012 Liberated Riesling (Arroyo Seco),US,This Riesling is distinctly off-dry. It's rich...,Liberated,85.0,16.0,California,Riesling,2012.0,thi riesl distinctli offdri rich sugari orang ...
60440,àMaurice 2014 Boushey Vineyard Marsanne-Viogni...,US,"The aromas of flowers, mineral, peach and almo...",Boushey Vineyard,89.0,35.0,Washington,Marsanne-Viognier,2014.0,aroma flower miner peach almond initi light pl...
49292,Scratch 2011 Riesling (Arroyo Seco),US,Unmistakably Riesling from the petrol notes th...,None,88.0,25.0,California,Riesling,2011.0,unmistak riesl petrol note domin winemak call ...


In [108]:
print(f'Top 3 recommendations for "{user_input}":\n')
for i in range(3):
    print(recommendations.iloc[i,0])
    print(recommendations.iloc[i,2])
    print()

Top 3 recommendations for "white flowers, mineral, peach":

Cascata 2006 Cascade Riesling (Finger Lakes)
This wine offers aromas of white flowers, spice and citrus. On the palate are easygoing flavors of white fruit, flowers and minerally spice. A simply good white wine with a feminine touch.

Dowsett Family 2014 Aunt Diane's Vineyard Riesling (Columbia Gorge (WA))
Perfumed aromas offer notes of white flowers, mineral and pink grapefruit. The flavors are dry and elegantly styled with a lingering finish. It will drink well at the dinner table.

Carmel Road 2012 Liberated Riesling (Arroyo Seco)
This Riesling is distinctly off-dry. It's rich with sugary orange and lime jam, white flower and mineral flavors, with the brisk acidity associated with the Arroyo Seco region. Easy to drink now.

